# 04 · Precios: validación de la conversión a kilos, P1 y P5

**Objetivo:** validar `precio_kg` (D-17), comparar la variabilidad de precios entre mercados (P1) y la relación entre calidad y precio (P5)
**Preguntas que responde:** P1, P5
**Entrada:** `data/processed/precios_limpio.parquet`
**Salida:** gráficos en `images/`
**Decisiones tomadas:** D-13, D-16, D-17, D-18 (las provisorias se cierran aquí)

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_parquet("../data/processed/precios_limpio.parquet")
print(df.shape)

# Mismo día, mercado, producto, variedad, calidad y origen, pero etiquetas distintas
clave = ["fecha", "mercado", "producto", "variedad", "calidad", "origen"]

d = df[df["precio_kg"].notna()].sort_values(clave + ["unidad"])
multi = d.duplicated(subset=clave, keep=False)
d2 = d[multi]

todos = d2.groupby(clave, observed=True).agg(
    n=("unidad", "nunique"),
    u1=("unidad", "first"), u2=("unidad", "last"),
    p1=("precio_kg", "first"), p2=("precio_kg", "last"),
)
print("Grupos con 2 o más etiquetas:", len(todos))
print(todos["n"].value_counts().sort_index())

# Solo grupos con exactamente 2 etiquetas (los de 3 o más quedan fuera de esta prueba)
par = todos[todos["n"] == 2].copy()
par["u1"] = par["u1"].astype(str)
par["u2"] = par["u2"].astype(str)

def forma(u):
    return re.sub(r"\d+(?:,\d+)?", "#", u)   # "$/caja 15 kilos" -> "$/caja # kilos"

f = {u: forma(u) for u in set(par["u1"]) | set(par["u2"])}
par["mismo_envase"] = par["u1"].map(f) == par["u2"].map(f)
par["logr"] = np.log(par["p1"] / par["p2"])
par["dif_pct"] = (np.exp(par["logr"].abs()) - 1) * 100   # diferencia % entre los dos precios por kilo

tabla = par.groupby("mismo_envase")["dif_pct"].agg(
    pares="size",
    mediana="median",
    p75=lambda s: s.quantile(.75),
    p90=lambda s: s.quantile(.90),
    dentro_10=lambda s: (s <= 10).mean() * 100,
    dentro_25=lambda s: (s <= 25).mean() * 100,
)
print(tabla.round(1))

(1297334, 21)
Grupos con 2 o más etiquetas: 30333
n
2    28570
3     1682
4       78
5        3
Name: count, dtype: int64
              pares  mediana   p75   p90  dentro_10  dentro_25
mismo_envase                                                  
False         22406     13.7  25.0  42.9       37.1       75.6
True           6164      6.1  12.5  23.5       66.0       91.5


In [2]:
por_par = (par.groupby(["mismo_envase", "u1", "u2"])
              .agg(n=("logr", "size"),
                   ratio=("logr", lambda s: float(np.exp(s.median()))),
                   dif_mediana=("dif_pct", "median"))
              .reset_index())

for flag, titulo in [(True, "MISMO ENVASE, DISTINTO PESO"), (False, "ENVASES DISTINTOS")]:
    print("\n" + titulo)
    print(por_par[por_par["mismo_envase"] == flag].query("n >= 20")
          .sort_values("n", ascending=False).head(15)
          .drop(columns="mismo_envase").round(2).to_string(index=False))


MISMO ENVASE, DISTINTO PESO
                u1                 u2    n  ratio  dif_mediana
$/bandeja 18 kilos $/bandeja 20 kilos 3586   0.98         4.94
  $/malla 15 kilos   $/malla 25 kilos  691   1.06         9.38
$/bandeja 10 kilos $/bandeja 18 kilos  305   1.10        13.08
  $/malla 16 kilos   $/malla 18 kilos  162   0.98         3.96
  $/malla 15 kilos   $/malla 18 kilos  115   0.99         5.46
  $/malla 18 kilos   $/malla 25 kilos   90   0.94         8.00
   $/caja 15 kilos    $/caja 25 kilos   86   1.00        18.02
$/bandeja 18 kilos  $/bandeja 8 kilos   69   0.71        40.62
$/bandeja 10 kilos  $/bandeja 8 kilos   64   0.96         6.90
$/bandeja 10 kilos $/bandeja 15 kilos   64   1.10         9.55
  $/malla 16 kilos   $/malla 20 kilos   61   1.00         6.25
   $/caja 10 kilos    $/caja 15 kilos   54   1.08        16.67
  $/malla 15 kilos   $/malla 20 kilos   48   0.91        10.11
   $/caja 10 kilos    $/caja 18 kilos   46   0.85        27.29
  $/malla 17 kilos   $/mal

In [3]:
cov = df.groupby("producto", observed=True)["precio_kg"].apply(lambda s: s.notna().mean())
sel = cov[cov >= 0.8].index
print("Productos con precio por kilo en >=80% de sus filas:", len(sel), "de", len(cov))

x = df[df["producto"].isin(sel) & (df["anio"] <= 2025)].copy()
x["mes_id"] = x["fecha"].dt.to_period("M")
m = (x.groupby(["producto", "mercado", "mes_id"], observed=True)
       .agg(precio=("precio_kg", "median"), n=("precio_kg", "count")).reset_index())

ser = (m[m["n"] >= 5].groupby(["producto", "mercado"], observed=True)
         .size().rename("meses").reset_index())
print(ser["meses"].describe().round(1))
for k in [24, 36, 48, 60]:
    ok = ser[ser["meses"] >= k]
    mercs = ok.groupby("producto", observed=True).size()
    print(f">= {k} meses: {len(ok)} series | {ok['producto'].nunique()} productos | "
          f"{(mercs >= 3).sum()} con 3+ mercados")

Productos con precio por kilo en >=80% de sus filas: 53 de 82
count    438.0
mean      42.3
std       25.9
min        1.0
25%       18.0
50%       40.0
75%       72.0
max       72.0
Name: meses, dtype: float64
>= 24 meses: 305 series | 42 productos | 34 con 3+ mercados
>= 36 meses: 240 series | 38 productos | 25 con 3+ mercados
>= 48 meses: 204 series | 29 productos | 22 con 3+ mercados
>= 60 meses: 166 series | 26 productos | 18 con 3+ mercados
